In [1]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import random

# Load the local dataset 
DATA_DIR = "./data/plantvillage/color"

dataset = load_dataset("imagefolder", data_dir=DATA_DIR)
ds = dataset['train']

print(f"Total images: {len(ds)}")
print(f"Number of classes: {ds.features['label'].num_classes}")

c:\Users\HP\OneDrive\Desktop\AgriGuard\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total images: 54305
Number of classes: 38


In [2]:
# Create a DataFrame with image index and label
data = []
for i in range(len(ds)):
    data.append({
        'index': i,
        'label_id': ds[i]['label'],
        'label_name': ds.features['label'].int2str(ds[i]['label'])
    })

df = pd.DataFrame(data)

print("Class distribution (top 10):")
print(df['label_name'].value_counts().head(10))

# Check imbalance
print(f"\nMost images in one class: {df['label_name'].value_counts().max()}")
print(f"Least images in one class: {df['label_name'].value_counts().min()}")

Class distribution (top 10):
label_name
Orange___Haunglongbing_(Citrus_greening)         5507
Tomato___Tomato_Yellow_Leaf_Curl_Virus           5357
Soybean___healthy                                5090
Peach___Bacterial_spot                           2297
Tomato___Bacterial_spot                          2127
Tomato___Late_blight                             1909
Squash___Powdery_mildew                          1835
Tomato___Septoria_leaf_spot                      1771
Tomato___Spider_mites Two-spotted_spider_mite    1676
Apple___healthy                                  1645
Name: count, dtype: int64

Most images in one class: 5507
Least images in one class: 152


In [3]:
# First split: 80% train, 20% temporary
train_idx, temp_idx = train_test_split(
    df.index, 
    test_size=0.20, 
    stratify=df['label_id'], 
    random_state=42
)

# Second split: split the 20% into val + test (50-50)
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.50, 
    stratify=df.loc[temp_idx, 'label_id'], 
    random_state=42
)

print(f"Train size: {len(train_idx)}")
print(f"Validation size: {len(val_idx)}")
print(f"Test size: {len(test_idx)}")

Train size: 43444
Validation size: 5430
Test size: 5431


In [4]:
# Create new folders
base_dir = Path("./data/processed_plantvillage")
for split in ['train', 'val', 'test']:
    for class_name in ds.features['label'].names:
        (base_dir / split / class_name).mkdir(parents=True, exist_ok=True)

print("Folder structure created!")

Folder structure created!


In [5]:
from PIL import Image

def copy_images(indices, split_name):
    for idx in indices:
        example = ds[int(idx)]
        label_name = ds.features['label'].int2str(example['label'])
        img = example['image']

        # Ensure image is in RGB mode
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Save image
        save_path = base_dir / split_name / label_name / f"{idx:06d}.jpg"
        img.save(save_path, format='JPEG', quality=95)  # specify JPEG format explicitly

print("Copying train images...")
copy_images(train_idx, 'train')

print("Copying val images...")
copy_images(val_idx, 'val')

print("Copying test images...")
copy_images(test_idx, 'test')

print(" All images copied successfully!")

Copying train images...
Copying val images...
Copying test images...
 All images copied successfully!
